# Assignment 6: Neural Network Showdown — Solution

## Setup

In [1]:
%pip install -q -r requirements.txt

# GPU acceleration (platform-specific)
import platform
if platform.system() == "Darwin" and platform.machine() == "arm64":
    %pip install -q tensorflow-metal

%reset -f

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import numpy as np
import pandas as pd

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Report available accelerators
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU acceleration: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"  {gpu.name}")
else:
    print("No GPU detected — using CPU")

from tensorflow import keras
from keras import Sequential
from keras.layers import (
    Dense, Flatten, Dropout, Conv2D, MaxPooling2D, LSTM, Input
)
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import confusion_matrix

from helpers import (
    load_cifar10, load_ecg5000,
    plot_training_history, plot_confusion_matrix,
    plot_sample_images, plot_ecg_traces, plot_predictions,
    CIFAR10_CLASSES, ECG_CLASSES,
)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete!")

GPU acceleration: 1 device(s)
  /physical_device:GPU:0
Setup complete!


---

## Part 1: Dense Baseline on CIFAR-10

In [3]:
print("Part 1: Dense Baseline on CIFAR-10")
print("-" * 40)

X_train, y_train, X_test, y_test = load_cifar10()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Part 1: Dense Baseline on CIFAR-10
----------------------------------------


Train: (50000, 32, 32, 3), Test: (10000, 32, 32, 3)


In [4]:
plot_sample_images(X_train, y_train, CIFAR10_CLASSES)

/Users/christopher/Documents/datasci_223/lectures/06/assignment/helpers.py:183: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
model_dense = Sequential([
    Input(shape=(32, 32, 3)),
    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax"),
])

In [6]:
model_dense.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [7]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history_dense = model_dense.fit(
    X_train, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
)

Epoch 1/20


  1/352 ━━━━━━━━━━━━━━━━━━━━ 2:59 513ms/step - accuracy: 0.1016 - loss: 3.2645

  6/352 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0994 - loss: 6.6554   

 12/352 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1014 - loss: 8.4252

 17/352 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1017 - loss: 9.5275

...

331/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2451 - loss: 2.1196

337/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2451 - loss: 2.1197

342/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2451 - loss: 2.1198

348/352 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2451 - loss: 2.1200

352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.2443 - loss: 2.1276 - val_accuracy: 0.2514 - val_loss: 2.0638


In [8]:
test_loss, test_acc = model_dense.evaluate(X_test, y_test, verbose=0)

y_pred = np.argmax(model_dense.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred)

In [9]:
results = {
    "accuracy": float(test_acc),
    "confusion_matrix": cm.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part1_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"Dense accuracy: {test_acc:.4f}")
print("Saved output/part1_results.json")

Dense accuracy: 0.2550
Saved output/part1_results.json


---

## Part 2: CNN on CIFAR-10

In [10]:
print("\nPart 2: CNN on CIFAR-10")
print("-" * 40)


Part 2: CNN on CIFAR-10
----------------------------------------


In [11]:
model_cnn = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),
])

In [12]:
model_cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [13]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    ModelCheckpoint(
        "output/best_cnn.keras",
        save_best_only=True,
        monitor="val_accuracy",
    ),
]

history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
)

Epoch 1/15


  1/704 ━━━━━━━━━━━━━━━━━━━━ 7:44 661ms/step - accuracy: 0.0625 - loss: 2.4084

  5/704 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.0651 - loss: 2.3870  

 10/704 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.0712 - loss: 2.3822 

 14/704 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.0754 - loss: 2.3803

...

688/704 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5028 - loss: 1.5777

693/704 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5028 - loss: 1.5783

698/704 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5027 - loss: 1.5790

703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5026 - loss: 1.5797

704/704 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4915 - loss: 1.6745 - val_accuracy: 0.4596 - val_loss: 1.9071


In [14]:
plot_training_history(
    history_cnn, os.path.join(OUTPUT_DIR, "part2_training_history.png")
)

In [15]:
cnn_loss, cnn_acc = model_cnn.evaluate(X_test, y_test, verbose=0)

y_pred_cnn = np.argmax(model_cnn.predict(X_test, verbose=0), axis=1)
y_true_cnn = np.argmax(y_test, axis=1)
cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)

In [16]:
results_cnn = {
    "accuracy": float(cnn_acc),
    "confusion_matrix": cm_cnn.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part2_results.json"), "w") as f:
    json.dump(results_cnn, f, indent=2)

comparison = pd.DataFrame([
    {"model": "Dense", "accuracy": float(test_acc)},
    {"model": "CNN", "accuracy": float(cnn_acc)},
])
comparison.to_csv(os.path.join(OUTPUT_DIR, "part2_comparison.csv"), index=False)

print(f"CNN accuracy:   {cnn_acc:.4f}")
print(f"Dense accuracy: {test_acc:.4f}")
print(f"Improvement:    {cnn_acc - test_acc:+.4f}")
print("Saved output/part2_results.json and output/part2_comparison.csv")

CNN accuracy:   0.5852
Dense accuracy: 0.2550
Improvement:    +0.3302
Saved output/part2_results.json and output/part2_comparison.csv


---

## Part 3: LSTM on ECG5000

In [17]:
print("\nPart 3: LSTM on ECG5000")
print("-" * 40)

X_train_ecg, y_train_ecg, X_test_ecg, y_test_ecg = load_ecg5000()
print(f"Train: {X_train_ecg.shape}, Test: {X_test_ecg.shape}")
print(f"Classes: {list(ECG_CLASSES.values())}")


Part 3: LSTM on ECG5000
----------------------------------------
Train: (4000, 140, 1), Test: (1000, 140, 1)
Classes: ['Normal', 'Supraventricular', 'Premature Ventricular', 'Fusion', 'Unknown']


In [18]:
plot_ecg_traces(X_train_ecg, y_train_ecg, ECG_CLASSES)

/Users/christopher/Documents/datasci_223/lectures/06/assignment/helpers.py:211: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
model_lstm = Sequential([
    Input(shape=(140, 1)),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(5, activation="softmax"),
])

In [20]:
model_lstm.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

In [21]:
early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train_ecg, y_train_ecg,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
)

Epoch 1/30


  1/113 ━━━━━━━━━━━━━━━━━━━━ 1:04 576ms/step - accuracy: 0.4375 - loss: 1.4952

  4/113 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5013 - loss: 1.4751   

  7/113 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5568 - loss: 1.4461

 10/113 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.6002 - loss: 1.4127

...

103/113 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9533 - loss: 0.1592

106/113 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9533 - loss: 0.1594

109/113 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9532 - loss: 0.1596

112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9532 - loss: 0.1597

113/113 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9508 - loss: 0.1667 - val_accuracy: 0.9225 - val_loss: 0.2562


In [22]:
plot_training_history(
    history_lstm, os.path.join(OUTPUT_DIR, "part3_training_history.png")
)

In [23]:
lstm_loss, lstm_acc = model_lstm.evaluate(X_test_ecg, y_test_ecg, verbose=0)

y_pred_ecg = np.argmax(model_lstm.predict(X_test_ecg, verbose=0), axis=1)
y_true_ecg = np.argmax(y_test_ecg, axis=1)
cm_ecg = confusion_matrix(y_true_ecg, y_pred_ecg)

In [24]:
results_ecg = {
    "accuracy": float(lstm_acc),
    "confusion_matrix": cm_ecg.tolist(),
}
with open(os.path.join(OUTPUT_DIR, "part3_results.json"), "w") as f:
    json.dump(results_ecg, f, indent=2)

print(f"LSTM accuracy: {lstm_acc:.4f}")
print("Saved output/part3_results.json")

LSTM accuracy: 0.9310
Saved output/part3_results.json


---

## Validation

In [25]:
print("\nAll parts complete!")
print("Run 'pytest .github/tests/ -v' in your terminal to check your work.")


All parts complete!
Run 'pytest .github/tests/ -v' in your terminal to check your work.
